# Import libraries

In [ ]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

import microspy as ms

# Specify file to load
Here we load the csv file written by Jeol's particle analysis software

In [ ]:
filename = "test_file.csv"

#### Load file
We might get some warnings if there are certain features that are not yet supported/fixed.

In [ ]:
s = ms.load(filename)

Print what's inside the vairable

In [ ]:
s

We see that ...
* The variable is a 'Particle analysis' type signal
* It's title
* The number of investigated particles are given by the number in the paranthesis.

### Let's check the signal's metadata:

In [ ]:
s.metadata

In [ ]:
s.metadata.get_item("Signals")

We identify two signals: Chemistry and Geometry

In [ ]:
s.Chemistry

In [ ]:
s.Geometry

Let's print the information in the two respective class attributes using the built-in function 'tabulate_data'

In [ ]:
ms.misc.tabulate_data(
    data = s.Chemistry.data,
    headers = s.elements, # A "shortcut" to the information in s.Chemistry.metadata.Signal.props
    unit = s.Chemistry.unit # Printed in the upper left corner
)

In [ ]:
ms.misc.tabulate_data(
    data = s.Geometry.data,
    headers = s.Geometry.metadata.Signal.props,
)

### We can also visualise the loaded data
If we only use the inherited function 'plot', no markers (here: the elements) will be overlayed.

In [ ]:
s.Chemistry.plot_with_markers()

# Load acquired images
Depending on the vendor, using 'load_images' might not work without specifying a directory. 

For the csv format, no directory is needed if the csv file is stored with the folder named as "Sutb$[EXPERIMENT NUMBER]$"

In [ ]:
s.load_images(
 **{
    "set_dtype" : np.uint8
    "read_CompositeSig" : True
    }
)

If we reprint the variable's information, we see that additional Images signals have been allocated to the variable: 

* (number of SEM images | SEM image shape | Number of particles)

The image can be accessed through the new attribute '$Images$'. The images are
* CompositeSig: A single image of individually stitched images
* ParentSig: Individual SEM images
* ChildSig: Individual particle images cropped from the ParentSig

In [ ]:
s.Images

We see that all Images' signals are 3D arrays except the CompositeSig. 

Calibrate the images and set the unit by using greek symbols from the microspy dictionary 'GREEK_LETTERS'

In [ ]:
s.calibrate_navigation(
    scale = 0.125, 
    unit= f"{ms.misc.GREEK_LETTERS.get("mu")}m"
)

We now have access to even more information:

In [ ]:
s

## Gridify images
The loaded images from Jeol's solution comes as a 3D array. If we want to specify the grid in which the images were acquired in, we can 'gridify' the signal.

Note!
The grid shape is (rows, cols)

In [ ]:
NUM_ROWS, NUM_COLS = 4, 5
s.gridify_ParentSig(
    navigation_shape=(NUM_ROWS, NUM_COLS)
)

The information is now updated:

In [ ]:
s

## Map particles' position
We know we have individual images of the particles, and the SEM images from where they were cropped from. But we don't have a map of their specific locations yet. Let's make it:

In [ ]:
s.map_particles()

The regions where the different particles are located is now stored in a new 'Images' class: 'ChildMap':

In [ ]:
s.Images

## Visualise
Let's visualise the result by highlighting these SEM regions:

In [ ]:
(s.Images.ParentSig * (s.Images.ChildMap > -1)).plot()

If we want to make a single phase map image, we can stitch the individual SEM images.

Note that we ought to know the image acquisition directions in advance, as different vendors might have different solutions on how they acquire and save the data.
For our case, the data has been acquired from top to bottom, and right to left:

In [ ]:
stitched_image = ms.draw.stitch_grid_signal(
    s.Images.ParentSig.data,
    grid_shape = (NUM_ROWS, NUM_COLS),
    horisontal_direction="r2l" # right -> left
)

Let's visualise the result by comparing the stitched image with the one saved by the vendor:

The stripes in the CompositeSig signal is probably highlighting sample drift.

In [ ]:
fig, ax = plt.subplots(2,1)

ax[0].imshow(s.Images.CompositeSig.data, cmap = "grey")
ax[0].set_title("Vendor stitched")

ax[1].imshow(stitched_image, cmap = "grey")
ax[1].set_title("Stitched parent signal")

for ax in ax.ravel(): ax.axis("off")
plt.tight_layout()
plt.show()

We can also create a mask that visually shows us where the stitching has taken place. That way, we might identify particles that have been cropped during the data acquisition.

In [ ]:
grid_mask = ms.draw.get_grid_mask(
    grid_shape = s.Images.ParentSig.data.shape 
)

plt.figure()
plt.imshow(
    stitched_image * grid_mask, 
    cmap = "grey"
)
plt.axis("off")
plt.show()
plt.tight_layout()

# Save data
Currently we can save the data both into Jeol's csv format, and microspy's hdf5 format. The former only saves the information about the particles. The latter saves this and the images into one single file.

We use the object's 'save' function

In [ ]:
s.save("inspected.hdf5")